# SpliceAI

![SpliceAI](https://proto-bio.github.io/proto-assets/images/tool/spliceai/hero.png)

SpliceAI is a deep neural network that predicts splicing from primary DNA sequence. It serves two purposes: scoring genetic variants for their splice-altering effect via delta scores (probabilities that the variant gains or loses an acceptor/donor site), and predicting raw per-position acceptor/donor splice-site probabilities directly from a DNA sequence. It is widely used to interpret the splicing impact of non-coding and synonymous variants in clinical and research genomics.

- Paper: [Jaganathan et al., Cell 2019](https://doi.org/10.1016/j.cell.2018.12.015)
- Repository: [Illumina/SpliceAI](https://github.com/Illumina/SpliceAI)

In [ ]:
from proto_tools.tools import (
    run_spliceai_score,
    run_spliceai_predict,
    SpliceAIScoreInput,
    SpliceAIScoreConfig,
    SpliceAIVariant,
    SpliceAIPredictInput,
    SpliceAIPredictConfig,
)
from proto_tools.utils.notebook_docs import display_api_reference

In [ ]:
# API reference for the variant-scoring tool
display_api_reference("spliceai-score", "input", "run_spliceai_score")
display_api_reference("spliceai-score", "config", "run_spliceai_score")
display_api_reference("spliceai-score", "output", "run_spliceai_score")

In [ ]:
# API reference for the raw splice-site prediction tool
display_api_reference("spliceai-predict", "input", "run_spliceai_predict")
display_api_reference("spliceai-predict", "config", "run_spliceai_predict")
display_api_reference("spliceai-predict", "output", "run_spliceai_predict")

## Variant scoring

Variant scoring needs a reference genome FASTA and a gene annotation: SpliceAI extracts the wild-type sequence window around each variant from the genome, applies the alternate allele, and compares the ensemble's predictions to derive delta scores for every overlapping gene. Here we bundle the real **HBB locus** (GRCh38 chr11, in transcription orientation) with a matching HBB annotation, and score **IVS-I-1** (`HBB:c.92+1G>A`) — the classic β-thalassemia variant that abolishes the intron-1 donor site. Real analyses use the full human genome (`grch37`/`grch38`) and the bundled GENCODE annotation.

In [ ]:
config = SpliceAIScoreConfig(
    reference_fasta="hbb_locus.fasta",
    annotation="hbb_annotation.txt",
    device="cpu",  # CPU so this runs on hosts without a GPU
)
# IVS-I-1 (HBB:c.92+1G>A): the classic beta-thalassemia variant that abolishes the
# HBB intron-1 donor splice site.
inputs = SpliceAIScoreInput(
    variants=[SpliceAIVariant(chromosome="HBB", position=5143, ref="G", alt="A")]
)

result = run_spliceai_score(inputs, config)

variant_result = result.results[0]
print(f"Variant: {variant_result.chromosome}:{variant_result.position} {variant_result.ref}>{variant_result.alt}")
print(f"Max delta score: {variant_result.metrics['max_delta_score']}")
print("\nPer-gene delta scores:")
for gene in variant_result.scores:
    print(
        f"  {gene.symbol} (allele {gene.allele}): "
        f"AG={gene.ds_ag:.3f} AL={gene.ds_al:.3f} "
        f"DG={gene.ds_dg:.3f} DL={gene.ds_dl:.3f}"
    )

## Raw splice-site prediction

Raw prediction runs directly on a DNA sequence and needs no reference genome. SpliceAI pads 5000 bp of context on each side internally, so the output covers every position of the input sequence with a `[neither, acceptor, donor]` probability triple.

In [ ]:
from pathlib import Path
import numpy as np

# Real HBB locus (GRCh38 chr11); SpliceAI flags the HBB exon/intron boundaries.
sequence = "".join(l for l in Path("hbb_locus.fasta").read_text().splitlines() if not l.startswith(">"))
predict_result = run_spliceai_predict(
    SpliceAIPredictInput(sequences=[sequence]),
    SpliceAIPredictConfig(device="cpu"),
)

probs = np.array(predict_result.predictions[0])  # channels: [neither, acceptor, donor]
print(f"Prediction shape: {probs.shape}  # [len, 3] -> [neither, acceptor, donor]")
print(f"Sequence length: {len(sequence)}")
print(f"Max acceptor probability (channel 1): {probs[:, 1].max():.4f} at position {probs[:, 1].argmax()}")
print(f"Max donor probability (channel 2): {probs[:, 2].max():.4f} at position {probs[:, 2].argmax()}")

In [ ]:
# Export the variant scores. Supported formats are listed by the output object.
import tempfile
from pathlib import Path

print(f"Supported formats: {result.output_format_options} (default: {result.output_format_default})")

with tempfile.TemporaryDirectory() as tmp_dir:
    result.export("spliceai_scores", export_path=tmp_dir, file_format="vcf")
    exported = Path(tmp_dir) / "spliceai_scores.vcf"
    print(f"Exported to {exported}")
    print(exported.read_text())